# Extensión: Diferenciación numérica

In [1]:
import numpy as np
import matplotlib.pyplot as plt

# Cargar los datos del Ejercicio 2
data = np.loadtxt("../datos/datos_sensor.csv", delimiter=",", skiprows=1)
x = data[:, 0]
y = data[:, 1]
h = x[1] - x[0]
n = len(x)

print(f"n = {n} datos, h = {h}")

n = 50 datos, h = 0.2


## 1. Forward difference

In [2]:
# Forward: (y[i+1] - y[i]) / h, para i=0..n-2
d_forward = np.diff(y) / h

# Corresponde a los puntos x[0] .. x[n-2] (49 valores)
x_forward = x[:-1]

print(f"Forward: {len(d_forward)} valores calculados")
print(f"Rango de x: {x_forward[0]:.2f} a {x_forward[-1]:.2f}")
print(f"Primeros 5: {d_forward[:5]}")

Forward: 49 valores calculados
Rango de x: 0.00 a 9.60
Primeros 5: [ 0.209015  0.085205 -0.011275 -0.06511  -0.06909 ]


## 2. Central difference

In [3]:
# Central: (y[i+1] - y[i-1]) / (2h), para i=1..n-2
d_central = (y[2:] - y[:-2]) / (2*h)

# Corresponde a los puntos x[1] .. x[n-2] (48 valores)
x_central = x[1:-1]

print(f"Central: {len(d_central)} valores calculados")
print(f"Rango de x: {x_central[0]:.2f} a {x_central[-1]:.2f}")
print(f"Primeros 5: {d_central[:5]}")

Central: 48 valores calculados
Rango de x: 0.20 a 9.60
Primeros 5: [ 0.14711    0.036965  -0.0381925 -0.0671    -0.0472125]


## 3. Tratamiento de los extremos

| Punto | Forward | Central |
|---|---|---|
| $i=0$ (primero) |  disponible |  falta $x_{-1}$ |
| $i=n-1$ (último) |  falta $x_{n}$ |  falta $x_{n}$ |
| Puntos interiores | disponible | disponible |

**Soluciones comunes para los extremos:**
- Usar **forward** en el primero y **backward** $(y_i - y_{i-1})/h$ en el último.
- Ajustar un polinomio local (interpolación) en los extremos.
- Dejar los extremos como NaN y analizar solo los puntos interiores.

## 5. Comparación con la derivada analítica

La función generadora es $y(x) = 2 + 0.35\sin(0.7x) + 0.15\cos(2.1x) + 0.03x$.

Su derivada analítica:
$$y'(x) = 0.245\cos(0.7x) - 0.315\sin(2.1x) + 0.03$$

In [4]:
# Derivada analítica (solo para verificar la precisión)
def dydx_real(x):
    return 0.245*np.cos(0.7*x) - 0.315*np.sin(2.1*x) + 0.03

# Errores
err_forward = np.abs(d_forward - dydx_real(x_forward))
err_central = np.abs(d_central - dydx_real(x_central))

print(f"Error máximo forward: {err_forward.max():.3e}")
print(f"Error máximo central: {err_central.max():.3e}")
print()
print(f"Error promedio forward: {err_forward.mean():.3e}")
print(f"Error promedio central: {err_central.mean():.3e}")
print()
print(f"El central es ~{err_forward.mean()/err_central.mean():.1f}× más preciso")

Error máximo forward: 8.073e-02
Error máximo central: 9.848e-03

Error promedio forward: 4.287e-02
Error promedio central: 5.852e-03

El central es ~7.3× más preciso


## 6. Sensibilidad al ruido

In [5]:
np.random.seed(42)   # reproducibilidad
sigma = 0.01
ruido = np.random.normal(0, sigma, size=len(y))
y_ruidoso = y + ruido

# Integral con y limpio vs ruidoso
from scipy.integrate import trapezoid
I_limpio   = trapezoid(y, x)
I_ruidoso  = trapezoid(y_ruidoso, x)

# Derivada central con y limpio vs ruidoso
d_central_limpio  = (y[2:] - y[:-2]) / (2*h)
d_central_ruidoso = (y_ruidoso[2:] - y_ruidoso[:-2]) / (2*h)

print("=== Efecto del ruido ===")
print()
print(f"Integral limpia:      {I_limpio:.6f}")
print(f"Integral con ruido:   {I_ruidoso:.6f}")
print(f"Error relativo:       {abs(I_ruidoso - I_limpio)/abs(I_limpio):.3e}")
print()
print(f"Derivada central limpia, rango: [{d_central_limpio.min():.3f}, {d_central_limpio.max():.3f}]")
print(f"Derivada central ruidosa, rango: [{d_central_ruidoso.min():.3f}, {d_central_ruidoso.max():.3f}]")
print()

# Error RMS de la derivada
rms_dif = np.sqrt(np.mean((d_central_ruidoso - d_central_limpio)**2))
print(f"RMS de la diferencia en derivada: {rms_dif:.3e}")

=== Efecto del ruido ===

Integral limpia:      21.190846
Integral con ruido:   21.169565
Error relativo:       1.004e-03

Derivada central limpia, rango: [-0.490, 0.544]
Derivada central ruidosa, rango: [-0.504, 0.529]

RMS de la diferencia en derivada: 3.331e-02


## 7. ¿Por qué la derivada es más sensible al ruido?

**Integración (trapecio):**
$$I \approx h \sum_i y_i \quad\Rightarrow\quad \Delta I \sim h\sqrt{n}\,\sigma$$

Los errores aleatorios **se cancelan parcialmente** al sumar (crecen como $\sqrt{n}$, no como $n$).

**Derivada (central):**
$$f'(x_i) \approx \frac{y_{i+1}-y_{i-1}}{2h} \quad\Rightarrow\quad \Delta f' \sim \frac{\sigma}{h}$$

Aquí los errores **se amplifican por $1/h$** y no se cancelan (son diferencias).

**Conclusión:** integrar **promedia** y atenúa el ruido; derivar **amplifica**. Es una consecuencia directa del teorema fundamental del cálculo.